# 08 · Baselines and outside geocoders (CPU)

**What this does:** runs everything that is *not* one of our trained models over the test sets and stores the answers.

| Group | Members | Rows sent | Cost |
|---|---|---|---|
| Local baselines | city centre, pincode centre, fuzzy gazetteer rules, nearest training address (TF-IDF) | all test rows | free |
| Open geocoders | Nominatim and Photon (both built on OpenStreetMap) | the API subset (about 2,500) | free, 1 request per second |
| Paid geocoders | Google Geocoding, Ola Maps, Mappls (MapmyIndia) | the API subset | inside each free tier if you run it once |

**Rules for paid geocoders. These come from their terms of service.**
1. Keys live in Colab or Kaggle *secrets* (`GOOGLE_MAPS_KEY`, `OLA_MAPS_KEY`, `MAPPLS_KEY`), never in the notebook or the repo.
2. Only **scores** are saved to the results folder: the error in metres and yes/no flags. Their coordinates are never published and never drawn on our map.
3. Their answers are never used to train, tune or validate any model.
4. The resume-cache that holds raw answers is private and is purged after 30 days (last cell).
5. When results are published, the write-up must give everything needed to reproduce them: date, endpoint, parameters, the exact test rows.

A provider with no key set is skipped, so the notebook runs end to end with whatever you have.

**Runtime:** local baselines about 5 minutes. Nominatim and Photon about 45 minutes each (rate limit). Paid ones about 10 minutes each.

**Needs:** notebook 06 (07 optional). The response parsers were written from each vendor's documentation and have not been run against live keys: section 3 checks each provider on three addresses before spending quota.

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz', 'scikit-learn']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/content/Navigate' if os.path.exists('/content') else '/kaggle/working/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

In [ ]:
from bench import baselines as bl, context as cx, geocoders as gc, io, metrics as mt, pincode as pc
B, G, RUNS = P['bench'], P['gaz'], P['runs']
test    = io.read(B, 'test')
train   = io.read(B, 'train')
targets = pd.read_parquet(f'{B}/targets.parquet')
lm      = pd.read_parquet(f'{B}/landmarks_clean.parquet')
pin_lookup, _ = pc.make_lookup(G)
api = test[test.api_subset]
print(f"{len(test):,} test rows, {len(api):,} in the API subset")

## 1. Local baselines

In [ ]:
pins = bl.pincode_table(targets[targets.split == 'train'])
roads = cx.load_roads(G); area_poly, localities, suburbs = cx.load_areas(G)
gaz = bl.build_gazetteer(lm, area_poly, localities, suburbs, roads)      # public landmarks only
print('gazetteer entries:', gaz.kind.value_counts().to_dict())

io.save_run(RUNS, 'city_centroid',    bl.city_centroid(test), about='Always answers the city centre.')
io.save_run(RUNS, 'pincode_centroid', bl.pincode_centroid(test, pins), about='Reads the pincode, answers its centre; city centre if there is none.')
io.save_run(RUNS, 'gazetteer_rules',  bl.GazetteerGeocoder(gaz, pins).geocode(test), about='Fuzzy lookup of place names in the public gazetteer; no learning.')
io.save_run(RUNS, 'tfidf_knn',        bl.TfidfKnn().fit(train).geocode(test), about='Location of the most similar training address (character n-grams).')
print('saved 4 local baselines')

## 2. Open geocoders (no key needed)
Public Nominatim allows 1 request per second and asks for an identifying User-Agent, which `bench/geocoders.py` sets. Safe to stop and re-run: finished rows are cached.

In [ ]:
RUN_OPEN = os.environ.get('NAV_SKIP_NETWORK') is None      # set to False to skip
if RUN_OPEN:
    for name, about in [('nominatim', 'OpenStreetMap search (public server), raw address text.'),
                        ('photon', 'Photon search over OpenStreetMap (public server), raw address text.')]:
        gc.check_provider(name)
        pred = gc.run_provider(name, api, f"{P['cache']}/{name}.jsonl")
        io.save_run(RUNS, name, pred, kind='open', about=about)

## 3. Paid geocoders (skipped unless a key is set)
Each is first tried on three easy addresses. **Read that output.** If the parsed answer is `None` while the raw reply clearly has coordinates, the reply format differs from the documentation: fix the matching `parse_...` function in `bench/geocoders.py` before running the full subset.

In [ ]:
ABOUT = {'google': 'Google Geocoding API, raw address text, biased to India and the study area.',
         'ola': 'Ola Maps geocoding API, raw address text.', 'mappls': 'Mappls (MapmyIndia) geocoding API, raw address text.'}
for name in ('google', 'ola', 'mappls'):
    key = env.get_secret(gc.PROVIDERS[name]['secret'])
    if not key:
        print(f"{name}: no key in secret {gc.PROVIDERS[name]['secret']}, skipped"); continue
    gc.check_provider(name, key)
    pred = gc.run_provider(name, api, f"{P['cache']}/{name}.jsonl", key=key)
    truth = api.reset_index(drop=True)
    scores = mt.score_rows(truth, pred, pin_lookup, targets)        # scored in memory ...
    io.save_run(RUNS, name, scores=gc.scores_only(scores, pred), kind='paid', about=ABOUT[name] + f" Run on {pd.Timestamp.now():%Y-%m-%d}.")
    del pred                                                          # ... and the coordinates are dropped
    print(name, 'median error on answered rows:', scores.err_m.median(), 'm')

## 4. Housekeeping: purge old cached answers from paid providers
Run this whenever you open the notebook. Anything older than 30 days is deleted.

In [ ]:
for name in ('google', 'ola', 'mappls'):
    path = f"{P['cache']}/{name}.jsonl"
    if os.path.exists(path): print(name, 'purged', gc.purge_cache(path, max_age_days=30), 'old answers')

## Share the outputs
Packs only the files this notebook created into one zip in `My Drive > Navigate > data > share`. Download that one file to share or back up the results.

In [ ]:
env.pack_outputs(P, '08' + '_outputs', [P['runs']])